# Example 2 - Strategy Analysis

Run a small file-free strategy loop with ranking, execution, turnover and returns.


In [ ]:
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.execution.naiveexecutor import NaiveExecutor
from alphamind.portfolio.rankbuilder import rank_build

executor = NaiveExecutor()
results = []
for trade_date in dates:
    section = panel.filter(pc.equal(panel["trade_date"], trade_date))
    signal = section["factor"].to_numpy(zero_copy_only=False)
    weights = rank_build(signal, use_rank=3).flatten().astype(float)
    weights /= weights.sum()
    target = pa.table({"code": section["code"], "weight": weights})
    turn_over, executed = executor.execute(target)
    executor.set_current(executed)
    portfolio_return = float(weights @ section["dx"].to_numpy(zero_copy_only=False))
    results.append(pa.table({
        "trade_date": [trade_date],
        "return": [portfolio_return],
        "turn_over": [turn_over],
    }))
result = pa.concat_tables(results)
assert result.num_rows == n_dates
result
